In [2]:
import sys
import numpy
import pandas
import sklearn
import lightgbm
import joblib
import scipy

print("=" * 60)
print("FRESH ENVIRONMENT INFORMATION")
print("=" * 60)

print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("Scikit-learn:", sklearn.__version__)
print("LightGBM:", lightgbm.__version__)
print("Joblib:", joblib.__version__)
print("SciPy:", scipy.__version__)

FRESH ENVIRONMENT INFORMATION
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
NumPy: 2.1.3
Pandas: 2.3.3
Scikit-learn: 1.6.1
LightGBM: 4.6.0
Joblib: 1.6.0
SciPy: 1.16.3


In [1]:
from pathlib import Path

print("AVAILABLE INPUT FILES")
print("=" * 60)

for path in Path("/kaggle/input").rglob("*"):
    if path.is_file():
        print(path)

AVAILABLE INPUT FILES
/kaggle/input/datasets/shizansarkar/shshsdda/reproduction_manifest.json
/kaggle/input/datasets/shizansarkar/sci-recovery-data/SCI_Neurological_Recovery-master/README.md
/kaggle/input/datasets/shizansarkar/sci-recovery-data/SCI_Neurological_Recovery-master/.Rhistory
/kaggle/input/datasets/shizansarkar/sci-recovery-data/SCI_Neurological_Recovery-master/Licence/LICENSE
/kaggle/input/datasets/shizansarkar/sci-recovery-data/SCI_Neurological_Recovery-master/EMSCI/.RData
/kaggle/input/datasets/shizansarkar/sci-recovery-data/SCI_Neurological_Recovery-master/EMSCI/.Rhistory
/kaggle/input/datasets/shizansarkar/sci-recovery-data/SCI_Neurological_Recovery-master/EMSCI/test.R
/kaggle/input/datasets/shizansarkar/sci-recovery-data/SCI_Neurological_Recovery-master/Formatting_data/format_datasets.R
/kaggle/input/datasets/shizansarkar/sci-recovery-data/SCI_Neurological_Recovery-master/test/spss_v1_2.csv
/kaggle/input/datasets/shizansarkar/sci-recovery-data/SCI_Neurological_Recovery

In [3]:

# ============================================================
# STEP 34.3: INDEPENDENT SAVED-MODEL REPRODUCTION
# NEW KAGGLE NOTEBOOK
# NO MODEL TRAINING
# ============================================================

import json
import hashlib
import zipfile
import sys
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import joblib
import sklearn
import lightgbm

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=" * 70)
print("STEP 34.3: CLEAN-ENVIRONMENT REPRODUCTION")
print("=" * 70)

INPUT_ROOT = Path("/kaggle/input")

OUTPUT_DIR = Path(
    "/kaggle/working/SCI_Fresh_Reproduction"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Find reproduction ZIP or extracted dataset
# ------------------------------------------------------------

zip_files = sorted(
    INPUT_ROOT.rglob(
        "SCI_Clean_Reproduction_Package.zip"
    )
)

manifest_files = sorted(
    INPUT_ROOT.rglob(
        "reproduction_manifest.json"
    )
)

if len(manifest_files) == 1:

    PACKAGE_ROOT = manifest_files[0].parent
    print("Using extracted package:", PACKAGE_ROOT)

elif len(zip_files) == 1:

    ZIP_PATH = zip_files[0]

    PACKAGE_ROOT = (
        OUTPUT_DIR / "extracted_package"
    )

    PACKAGE_ROOT.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(ZIP_PATH) as archive:

        for member in archive.infolist():

            target = (
                PACKAGE_ROOT / member.filename
            ).resolve()

            if not target.is_relative_to(
                PACKAGE_ROOT.resolve()
            ):
                raise RuntimeError(
                    "Unsafe ZIP file path detected."
                )

        archive.extractall(PACKAGE_ROOT)

    print("ZIP extracted successfully.")

else:

    raise FileNotFoundError(
        "Cannot identify exactly one SCI "
        "reproduction package in /kaggle/input."
    )

# ------------------------------------------------------------
# 2. Load manifest
# ------------------------------------------------------------

MANIFEST_PATH = (
    PACKAGE_ROOT /
    "reproduction_manifest.json"
)

manifest = json.loads(
    MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

# ------------------------------------------------------------
# 3. Locate required files
# ------------------------------------------------------------

files = {

    "model":
        PACKAGE_ROOT / "model" /
        "SCI_Dynamic_Recovery_LightGBM_Model.pkl",

    "raw_features":
        PACKAGE_ROOT / "model" /
        "SCI_Model_Feature_List.pkl",

    "processed_features":
        PACKAGE_ROOT / "model" /
        "SCI_Model_Processed_Feature_List.pkl",

    "X_test":
        PACKAGE_ROOT / "data" /
        "frozen_X_test.pkl",

    "y_test":
        PACKAGE_ROOT / "data" /
        "frozen_y_test.npy",

    "predictions":
        PACKAGE_ROOT / "data" /
        "original_predictions.npy"
}

# ------------------------------------------------------------
# 4. Verify SHA-256 integrity
# ------------------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with path.open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


for name, path in files.items():

    if not path.is_file():
        raise FileNotFoundError(
            f"Missing file: {path}"
        )

    actual_hash = sha256_file(path)

    expected_hash = (
        manifest["files"][name]["sha256"]
    )

    assert actual_hash == expected_hash, (
        f"Hash mismatch: {name}"
    )

print("\nAll six artifact hashes: PASSED")

# ------------------------------------------------------------
# 5. Load model and frozen patient data
# ------------------------------------------------------------

# Load only trusted artifacts from your own project.

model = joblib.load(files["model"])

raw_features = list(
    joblib.load(files["raw_features"])
)

processed_features = list(
    joblib.load(files["processed_features"])
)

X_test = pd.read_pickle(
    files["X_test"]
)

y_test = np.load(
    files["y_test"],
    allow_pickle=False
)

original_predictions = np.load(
    files["predictions"],
    allow_pickle=False
)

assert len(X_test) == 99
assert len(y_test) == 99
assert len(original_predictions) == 99

assert list(
    X_test.columns
) == raw_features

preprocessor = (
    model.named_steps["preprocessor"]
)

assert list(
    preprocessor.feature_names_in_
) == raw_features

assert list(
    preprocessor.get_feature_names_out()
) == processed_features

print("Raw feature order: PASSED")
print("Processed feature order: PASSED")

# ------------------------------------------------------------
# 6. Generate predictions
# ------------------------------------------------------------

new_predictions = np.asarray(
    model.predict(X_test),
    dtype=float
).ravel()

maximum_difference = float(
    np.max(
        np.abs(
            new_predictions -
            original_predictions
        )
    )
)

predictions_match = bool(
    np.allclose(
        new_predictions,
        original_predictions,
        atol=1e-5,
        rtol=0
    )
)

# ------------------------------------------------------------
# 7. Recalculate performance
# ------------------------------------------------------------

metrics = {

    "MAE": float(
        mean_absolute_error(
            y_test,
            new_predictions
        )
    ),

    "RMSE": float(
        np.sqrt(
            mean_squared_error(
                y_test,
                new_predictions
            )
        )
    ),

    "R2": float(
        r2_score(
            y_test,
            new_predictions
        )
    )
}

expected_metrics = (
    manifest["expected_metrics"]
)

metrics_match = bool(
    all(
        abs(
            metrics[key] -
            expected_metrics[key]
        ) <= 0.00015

        for key in [
            "MAE",
            "RMSE",
            "R2"
        ]
    )
)

verification_passed = bool(
    predictions_match and metrics_match
)

# ------------------------------------------------------------
# 8. Record environment information
# ------------------------------------------------------------

environment = {

    "python": sys.version,

    "platform": platform.platform(),

    "numpy": np.__version__,

    "pandas": pd.__version__,

    "scikit_learn": sklearn.__version__,

    "lightgbm": lightgbm.__version__,

    "joblib": joblib.__version__
}

# ------------------------------------------------------------
# 9. Save independent verification report
# ------------------------------------------------------------

report = {

    "verified_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "verification_type":
        "Independent saved-model inference reproduction",

    "patient_count": 99,

    "artifact_hashes_verified": True,

    "raw_feature_order_verified": True,

    "processed_feature_order_verified": True,

    "predictions_match": predictions_match,

    "metrics_match": metrics_match,

    "maximum_prediction_difference":
        maximum_difference,

    "reproduced_metrics": metrics,

    "environment": environment,

    "reproduction_checks_passed":
        verification_passed,

    "session_independence":
        "Requires researcher confirmation",

    "model_retrained": False
}

REPORT_PATH = (
    OUTPUT_DIR /
    "clean_session_reproduction_result.json"
)

REPORT_PATH.write_text(
    json.dumps(
        report,
        indent=4
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 10. Print verification results
# ------------------------------------------------------------

print("\nREPRODUCED TEST PERFORMANCE")

for key, value in metrics.items():
    print(f"{key}: {value:.6f}")

print(
    "\nMaximum prediction difference:",
    maximum_difference
)

print(
    "Predictions match:",
    predictions_match
)

print(
    "Metrics match:",
    metrics_match
)

print(
    "Independent reproduction checks passed:",
    verification_passed
)

print("\nReport saved:", REPORT_PATH)

if not verification_passed:

    raise AssertionError(
        "Reproduction verification failed."
    )

print("\nSTEP 34.3 COMPLETED SUCCESSFULLY")


STEP 34.3: CLEAN-ENVIRONMENT REPRODUCTION
Using extracted package: /kaggle/input/datasets/shizansarkar/shshsdda

All six artifact hashes: PASSED
Raw feature order: PASSED
Processed feature order: PASSED

REPRODUCED TEST PERFORMANCE
MAE: 5.542201
RMSE: 10.304732
R2: 0.880022

Maximum prediction difference: 0.0
Predictions match: True
Metrics match: True
Independent reproduction checks passed: True

Report saved: /kaggle/working/SCI_Fresh_Reproduction/clean_session_reproduction_result.json

STEP 34.3 COMPLETED SUCCESSFULLY


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [4]:
from IPython.display import FileLink, display
from pathlib import Path

file_path = Path(
    "/kaggle/working/SCI_Fresh_Reproduction/"
    "clean_session_reproduction_result.json"
)

assert file_path.exists(), "JSON file not found!"

display(FileLink(str(file_path)))

/kaggle/working/SCI_Fresh_Reproduction/clean_session_reproduction_result.json